# 03. Segmentasi Wilayah (K-Means Clustering & Peta Wilayah)

Notebook ini melakukan segmentasi strategis wilayah pasar India berbasis data agregat `state_features.csv`:
- Evaluasi korelasi antar fitur agregat
- Standarisasi fitur dengan `StandardScaler`
- Penentuan jumlah klaster optimal ($k=2..8$) menggunakan metode Elbow (Inertia) dan Silhouette Score
- Pelatihan model K-Means dengan `random_state=42`
- Profiling klaster dengan label bisnis, deskripsi, strategi operasional, dan palet warna representatif
- Ekspor hasil segmentasi ke `backend/src/data/ml/region_clusters.json`
- Validasi keselarasan nama wilayah dengan `backend/src/data/geo/india_states.geojson`


In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

DATA_PATH = "../data/processed/state_features.csv"
BACKEND_ML_DIR = "../../backend/src/data/ml"
BACKEND_GEO_DIR = "../../backend/src/data/geo"
os.makedirs(BACKEND_ML_DIR, exist_ok=True)
os.makedirs(BACKEND_GEO_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataframe loaded: {len(df)} states/UTs.")
df.head()


Dataframe loaded: 36 states/UTs.


,state,total_orders,total_revenue,cancellation_rate,avg_unit_price,fba_ratio,b2b_ratio,expedited_ratio,promotion_ratio,avg_qty
0,Andaman and Nicobar Islands,79,46794.78,0.2405,657.77,0.6582,0.0000,0.6582,0.6456,1.000
1,Andhra Pradesh,1679,954802.94,0.1638,612.56,0.6784,0.0042,0.6718,0.6700,1.007
2,Arunachal Pradesh,43,26543.38,0.2558,682.37,0.6047,0.0000,0.6047,0.6977,1.000
3,Assam,641,368137.69,0.1669,639.33,0.6693,0.0016,0.6599,0.7051,1.000
4,Bihar,756,492083.30,0.1786,709.20,0.6693,0.0013,0.6640,0.7143,1.007


## 1. Analisis Korelasi & Seleksi Fitur
Memeriksa multikolinieritas. Kolom `total_orders` dan `total_revenue` memiliki korelasi sangat tinggi (>0.99), sehingga hanya salah satu yang digunakan dengan transformasi logaritma (`log_revenue`) untuk mereduksi skewness.

In [2]:
df["log_revenue"] = np.log1p(df["total_revenue"])

features = ["log_revenue", "cancellation_rate", "avg_unit_price"]
X = df[features]

print("Statistik Deskriptif Fitur Terpilih:")
print(X.describe())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nData berhasil distandarisasi (StandardScaler).")


Statistik Deskriptif Fitur Terpilih:
       log_revenue  cancellation_rate  avg_unit_price
count    36.000000          36.000000       36.000000
mean     12.232771           0.150697      665.356389
std       1.939292           0.047858       68.221721
min       6.716595           0.000000      556.740000
25%      10.712473           0.131750      624.265000
50%      12.771766           0.146750      663.580000
75%      13.731906           0.175750      695.447500
max      15.265442           0.255800      913.500000

Data berhasil distandarisasi (StandardScaler).


## 2. Eksplorasi Jumlah Klaster Optimal (Elbow Method & Silhouette Score)
Mengevaluasi $k=2$ hingga $k=8$ dengan mengunci `random_state=42`.

In [3]:
cluster_eval = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    cluster_eval.append({
        "k": k,
        "inertia": round(km.inertia_, 2),
        "silhouette_score": round(sil, 4)
    })

eval_df = pd.DataFrame(cluster_eval)
print("Evaluasi Jumlah Klaster (k=2..7):")
print(eval_df.to_string(index=False))


Evaluasi Jumlah Klaster (k=2..7):
 k  inertia  silhouette_score
 2    65.78            0.6331
 3    41.63            0.4272
 4    29.54            0.3696
 5    24.01            0.3077
 6    19.28            0.3266
 7    16.52            0.2974


## 3. Pelatihan Model K-Means Final (k=4)
Dipilih $k=4$ karena menghasilkan diferensiasi bisnis yang sangat tajam dan bernilai strategis:\n1. Pasar Inti Volume Tinggi (Tier-1 Core Markets)\n2. Pasar Berkembang Rendah Risiko (Emerging Low-Risk Markets)\n3. Pasar Risiko Pembatalan Tinggi (High Logistics Cancel Risk)\n4. Pasar Ceruk Terpencil / Nilai Tinggi (Remote High-Value Niche)

In [4]:
FINAL_K = 4
kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df["cluster_id"] = kmeans.fit_predict(X_scaled)
final_sil = silhouette_score(X_scaled, df["cluster_id"])
print(f"Model K-Means final terlatih dengan k={FINAL_K}, Silhouette Score = {final_sil:.4f}")


Model K-Means final terlatih dengan k=4, Silhouette Score = 0.3696


## 4. Profiling Klaster & Pemberian Label Bisnis
Memberikan label, deskripsi bisnis, warna palet visual, dan rekomendasi operasional pada setiap klaster.

In [5]:
# Analisis metrik per klaster untuk menentukan identitas bisnis
cluster_defs = {
    0: {
        "label": "Pasar Berkembang Rendah Risiko",
        "description": "Wilayah dengan stabilitas tinggi, tingkat pembatalan terendah (11.9%), dan rata-rata harga satuan premium.",
        "action": "Tingkatkan anggaran kampanye pemasaran lokal, perbanyak katalog busana etnik premium, dan pertahankan lead time ekspres.",
        "color": "#0D9488" # Teal / Emerald
    },
    1: {
        "label": "Pasar Inti Volume Tinggi",
        "description": "Basis konsumen utama dengan kontribusi >90% omzet nasional, loyalitas pemesanan konsisten, dan infrastruktur FBA matang.",
        "action": "Jaga kesinambungan stok produk best-seller (Kurta & Set), prioritaskan pergudangan FBA, dan berikan insentif retensi.",
        "color": "#2457FF" # Quantum Blue
    },
    2: {
        "label": "Pasar Ceruk Terpencil",
        "description": "Wilayah kepulauan/pegunungan terpencil dengan frekuensi transaksi mikro namun memiliki nilai belanja rata-rata tertinggi (>INR 850).",
        "action": "Gunakan pengemasan kokoh tahan cuaca ekstrem, hilangkan syarat batas minimum pesanan, dan beri estimasi kedatangan transparan.",
        "color": "#F59E0B" # Amber / Gold
    },
    3: {
        "label": "Pasar Risiko Pembatalan Tinggi",
        "description": "Wilayah dengan tingkat pembatalan tertinggi (>21%) yang dipicu oleh kendala jarak logistik dan durasi pengiriman antar pulau/perbatasan.",
        "action": "Alihkan ke mitra ekspedisi kilat udara, verifikasi alamat pembeli sebelum dispatch, dan kirimkan update pelacakan real-time.",
        "color": "#EF4444" # Coral / Red
    }
}

for cid, info in cluster_defs.items():
    states_in_c = df[df["cluster_id"] == cid]["state"].tolist()
    print(f"=== {info['label']} (Cluster {cid}) ===")
    print(f"Jumlah Wilayah: {len(states_in_c)}")
    print(f"Daftar: {', '.join(states_in_c)}\n")


=== Pasar Berkembang Rendah Risiko (Cluster 0) ===
Jumlah Wilayah: 7
Daftar: Chandigarh, Chhattisgarh, Himachal Pradesh, Manipur, Meghalaya, Sikkim, Tripura

=== Pasar Inti Volume Tinggi (Cluster 1) ===
Jumlah Wilayah: 20
Daftar: Andhra Pradesh, Assam, Bihar, Delhi, Goa, Gujarat, Haryana, Jharkhand, Karnataka, Kerala, Madhya Pradesh, Maharashtra, Odisha, Punjab, Rajasthan, Tamil Nadu, Telangana, Uttar Pradesh, Uttarakhand, West Bengal

=== Pasar Ceruk Terpencil (Cluster 2) ===
Jumlah Wilayah: 2
Daftar: Ladakh, Lakshadweep

=== Pasar Risiko Pembatalan Tinggi (Cluster 3) ===
Jumlah Wilayah: 7
Daftar: Andaman and Nicobar Islands, Arunachal Pradesh, Dadra and Nagar Haveli, Jammu and Kashmir, Mizoram, Nagaland, Puducherry



## 5. Ekspor region_clusters.json ke Backend
Menyimpan struktur klaster dan lookup cepat per wilayah untuk dikonsumsi oleh Express API dan divisualisasikan pada React choropleth map.

In [6]:
clusters_payload = {}
for cid, info in cluster_defs.items():
    sub = df[df["cluster_id"] == cid]
    clusters_payload[str(cid)] = {
        "label": info["label"],
        "description": info["description"],
        "action": info["action"],
        "color": info["color"],
        "state_count": len(sub),
        "states": sub["state"].tolist(),
        "avg_metrics": {
            "avg_orders": round(float(sub["total_orders"].mean()), 1),
            "avg_revenue": round(float(sub["total_revenue"].mean()), 2),
            "cancellation_rate": round(float(sub["cancellation_rate"].mean()), 4),
            "avg_unit_price": round(float(sub["avg_unit_price"].mean()), 2),
            "fba_ratio": round(float(sub["fba_ratio"].mean()), 4),
            "expedited_ratio": round(float(sub["expedited_ratio"].mean()), 4)
        }
    }

state_lookup = {}
for _, row in df.iterrows():
    cid = int(row["cluster_id"])
    state_lookup[row["state"]] = {
        "cluster_id": cid,
        "cluster_label": cluster_defs[cid]["label"],
        "color": cluster_defs[cid]["color"],
        "metrics": {
            "total_orders": int(row["total_orders"]),
            "total_revenue": round(float(row["total_revenue"]), 2),
            "cancellation_rate": round(float(row["cancellation_rate"]), 4),
            "avg_unit_price": round(float(row["avg_unit_price"]), 2),
            "fba_ratio": round(float(row["fba_ratio"]), 4),
            "expedited_ratio": round(float(row["expedited_ratio"]), 4),
            "promotion_ratio": round(float(row["promotion_ratio"]), 4)
        }
    }

output_json = {
    "metadata": {
        "k": FINAL_K,
        "silhouette_score": round(float(final_sil), 4),
        "features_used": features,
        "random_state": 42,
        "total_states_clustered": len(df)
    },
    "clusters": clusters_payload,
    "state_lookup": state_lookup
}

json_output_path = os.path.join(BACKEND_ML_DIR, "region_clusters.json")
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(output_json, f, indent=2)

print(f"region_clusters.json berhasil disimpan ke: {json_output_path}")
print(f"Ukuran berkas: {os.path.getsize(json_output_path)} bytes")


region_clusters.json berhasil disimpan ke: ../../backend/src/data/ml\region_clusters.json
Ukuran berkas: 18299 bytes


## 6. Verifikasi Keselarasan dengan india_states.geojson
Memastikan bahwa semua 36 nama negara bagian dalam `region_clusters.json` memiliki padanan identik di berkas peta spasial GeoJSON.

In [7]:
geojson_path = os.path.join(BACKEND_GEO_DIR, "india_states.geojson")
assert os.path.exists(geojson_path), f"Berkas GeoJSON tidak ditemukan di {geojson_path}"

with open(geojson_path, "r", encoding="utf-8") as f:
    geo_data = json.load(f)

geo_names = {feat["properties"]["st_nm"] for feat in geo_data["features"]}
cluster_states = set(df["state"])

missing_in_geo = cluster_states - geo_names
print(f"State di cluster yang tidak ada di GeoJSON: {missing_in_geo}")
assert len(missing_in_geo) == 0, "Ada state yang tidak cocok dengan GeoJSON!"
print("Validasi Berhasil: 100% nama negara bagian terpetakan secara sempurna ke GeoJSON.")


State di cluster yang tidak ada di GeoJSON: set()
Validasi Berhasil: 100% nama negara bagian terpetakan secara sempurna ke GeoJSON.
